In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [2]:
df = pd.read_csv("sg_synthetic_training_data.csv")
df.head()

,customer_id,age,occupation,potentially_vulnerable_address,bankruptcy_or_iva,deposits_60d_gbp,losses_60d_gbp,lifetime_npr_gbp,failed_dep_insufficient_funds_24h,payment_methods_added_7d,...,tr_occ_unemployed,tr_occ_student,tr_occ_retired,tr_vulnerable_address,tr_limits_in_place_reduction,tr_reality_check,tr_sof_approved,sg_score,risk_band,high_risk_target
0,CUST_000001,42,retired,0,0,890.54,717.82,2007.18,0,0,...,0,0,1,0,1,0,0,-20,none_or_negative,0
1,CUST_000002,24,employed,0,0,376.13,237.91,223.87,0,0,...,0,0,0,0,1,0,1,-55,none_or_negative,0
2,CUST_000003,48,retired,0,0,1928.78,1356.80,569.50,1,0,...,0,0,1,0,1,0,0,-10,none_or_negative,0
3,CUST_000004,50,employed,1,0,3582.17,488.01,356.11,0,0,...,0,0,0,1,0,0,0,40,medium,0
4,CUST_000005,18,employed,0,0,912.77,1005.99,748.14,0,1,...,0,0,0,0,1,0,0,-15,none_or_negative,0


In [3]:
def find_col(keyword):
    for c in df.columns:
        if keyword in c.lower():
            return c
    return None

player_col = find_col("customer") or "customer_id"
deposit_col = find_col("deposit") or "deposits_60d_gbp"
loss_col = find_col("loss") or "losses_60d_gbp"
wager_col = find_col("wager") or "wagered_24h_gbp"

In [4]:
def safe_col(col, default=0):
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce").fillna(default)
    else:
        return pd.Series(default, index=df.index)

In [5]:
df["deposit_to_wager_ratio"] = safe_col(wager_col) / (safe_col(deposit_col) + 1)
df["loss_to_deposit_ratio"] = safe_col(loss_col) / (safe_col(deposit_col) + 1)
df["session_intensity"] = safe_col("active_hours_24h") / (safe_col("active_hours_7d", 1) + 1)

df["failed_dep_flag"] = (safe_col("failed_dep_insufficient_funds_24h") > 0).astype(int)
df["low_withdrawal_flag"] = (safe_col("withdrawal_within_24h") < 0.1).astype(int)
df["financial_stress_score"] = df["failed_dep_flag"] + df["low_withdrawal_flag"]

In [6]:
LOW, MEDIUM, HIGH, CRITICAL = 1, 2, 3, 5

behavior_weights = {
    "repeat_segment_entries_7d": LOW,
    "tr_extended_gameplay": MEDIUM,
    "tr_long_session": MEDIUM,
    "tr_at_risk_hours": HIGH,
    "tr_weekly_hours": MEDIUM,
    "tr_binges": HIGH,
    "tr_big_win_then_increase": CRITICAL,
    "tr_high_wagering": HIGH
}

for col, weight in behavior_weights.items():
    df[f"{col}_score"] = safe_col(col) * weight

df["behavioral_risk_raw"] = sum(df[f"{col}_score"] for col in behavior_weights)
max_behavior_score = sum(behavior_weights.values())
df["behavioral_risk_score"] = df["behavioral_risk_raw"] / max_behavior_score

In [7]:
financial_features = [
    "deposit_to_wager_ratio",
    "loss_to_deposit_ratio",
    "session_intensity",
    "financial_stress_score"
]

df_fin_norm = (df[financial_features] - df[financial_features].min()) / \
              (df[financial_features].max() - df[financial_features].min())

df["financial_risk_score"] = df_fin_norm.mean(axis=1)

In [8]:
df["early_risk_score"] = 0.5 * df["financial_risk_score"] + 0.5 * df["behavioral_risk_score"]

In [9]:
df["ml_label"] = (df["early_risk_score"] >= df["early_risk_score"].quantile(0.90)).astype(int)

In [10]:
features = [
    "financial_risk_score",
    "behavioral_risk_score",
    "deposit_to_wager_ratio",
    "loss_to_deposit_ratio",
    "session_intensity",
    "financial_stress_score",
] + list(behavior_weights.keys())

X = df[features]
y = df["ml_label"]

# Optional: scaling for Logistic Regression
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

# Save scaler
joblib.dump(scaler, "scaler_v1.pkl")

['scaler_v1.pkl']

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Save model
joblib.dump(model, "logistic_model_v1.pkl")

# Save feature list
joblib.dump(features, "model_features_v1.pkl")

['model_features_v1.pkl']